In [17]:
!pip install datasets evaluate sacrebleu rouge-score bert-score -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.0 MB/s eta 0:00:00


In [18]:
!pip install sentence-transformers -q

In [4]:
!pip install evaluate sacrebleu rouge-score meteor-score -q

ERROR: Could not find a version that satisfies the requirement meteor-score (from versions: none)
ERROR: No matching distribution found for meteor-score


In [1]:
from datasets import load_dataset

dataset = load_dataset("ai4bharat/IndicMTEval")

print(dataset)

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/498k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/200k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.13M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2380 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/4997 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['src', 'ref', 'translation', 'mqm_norm_score', 'da_norm_score', 'error_spans', 'language', 'split'],
        num_rows: 2380
    })
    validation: Dataset({
        features: ['src', 'ref', 'translation', 'mqm_norm_score', 'da_norm_score', 'error_spans', 'language', 'split'],
        num_rows: 1000
    })
    train: Dataset({
        features: ['src', 'ref', 'translation', 'mqm_norm_score', 'da_norm_score', 'error_spans', 'language', 'split'],
        num_rows: 4997
    })
})


In [5]:
print(dataset['test'][0])

{'src': 'The Chaco region was home to other groups of indigenous tribes such as the Guaycurú and Payaguá, who survived by hunting, gathering and fishing.', 'ref': 'ਚਾਕੋ ਖੇਤਰ ਦੇਸੀ ਕਬੀਲਿਆਂ ਦੇ ਹੋਰ ਸਮੂਹਾਂ ਜਿਵੇਂ ਗਾਇਕੂਰੁ ਅਤੇ ਪਾਇਗੂਆ ਦਾ ਘਰ ਸੀ, ਜੋ ਸ਼ਿਕਾਰ, ਇਕੱਠੇ ਰਹਿਣ ਅਤੇ ਫਿਸ਼ਿੰਗ ਦੁਆਰਾ ਬਚ ਗਏ ਸਨ।', 'translation': 'ਚਾਕੋ ਖੇਤਰ ਸਵਦੇਸ਼ੀ ਕਬੀਲਿਆਂ ਦੇ ਹੋਰ ਸਮੂਹਾਂ ਦਾ ਘਰ ਸੀ ਜਿਵੇਂ ਕਿ ਗੁਆਏਕੁਰੂ ਅਤੇ ਪਯਾਗੁਆ, ਜੋ ਸ਼ਿਕਾਰ, ਇਕੱਠੇ ਕਰਨ ਅਤੇ ਮੱਛੀਆਂ ਫੜ ਕੇ ਬਚੇ ਸਨ।', 'mqm_norm_score': '0.84', 'da_norm_score': '0.4', 'error_spans': [{'span_end_offset': 24, 'span_no': 0, 'span_severity': 'High', 'span_start_offset': 17, 'span_text': 'ਇਕੱਠੇ ਕਰਨ ਅਤੇ ਮੱਛੀਆਂ ਫੜ ਕੇ ਬਚੇ ਸਨ', 'span_type': 'Style_Awkward'}], 'language': 'Punjabi', 'split': 'test'}


In [6]:
dataset_small = dataset['test'].shuffle(seed=42).select(range(1000))

print(len(dataset_small))

1000


In [7]:
print(dataset_small[0])

{'src': 'There were no large forests in the land of Canaan, so wood was extremely expensive.', 'ref': 'கானானின் நாட்டில் பெரிய காடுகள் இல்லாததால், மரம் மிகவும் விலை உயர்ந்ததாய் இருந்தது.', 'translation': 'கானான் தேசத்தில் பெரிய காடுகள் இல்லை, எனவே\xa0மரம் மிகவும் விலை உயர்ந்தது.', 'mqm_norm_score': '0.92', 'da_norm_score': '0.92', 'error_spans': [{'span_end_offset': 9, 'span_no': 0, 'span_severity': 'Medium', 'span_start_offset': 6, 'span_text': 'மரம் மிகவும் விலை உயர்ந்தது', 'span_type': 'Style_Awkward'}], 'language': 'Tamil', 'split': 'test'}


In [8]:
sources = []
references = []
predictions = []

for example in dataset_small:
    sources.append(example['src'])
    references.append(example['ref'])
    predictions.append(example['translation'])

In [ ]:
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

references = [preprocess(t) for t in references]
predictions = [preprocess(t) for t in predictions]

In [ ]:
references_bleu = [[ref] for ref in references]

In [ ]:
import evaluate

bleu = evaluate.load("bleu")

bleu_score = bleu.compute(
    predictions=predictions,
    references=references_bleu
)

print("BLEU Score:", bleu_score["bleu"])

In [ ]:
rouge = evaluate.load("rouge")

rouge_score = rouge.compute(
    predictions=predictions,
    references=references
)

print("ROUGE Scores:", rouge_score)

In [ ]:
meteor = evaluate.load("meteor")

meteor_score = meteor.compute(
    predictions=predictions,
    references=references
)

print("METEOR Score:", meteor_score["meteor"])

In [ ]:
import sacrebleu

chrf = sacrebleu.corpus_chrf(predictions, [references])

print("chrF Score:", chrf.score)

In [ ]:
print("\n===== Machine Translation Evaluation =====")

print("BLEU:", bleu_score["bleu"])
print("ROUGE-L:", rouge_score["rougeL"])
print("METEOR:", meteor_score["meteor"])
print("chrF:", chrf.score)

In [ ]:
mqm_scores = [float(x) for x in dataset_small['mqm_norm_score']]
da_scores = [float(x) for x in dataset_small['da_norm_score']]

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
ref_embeddings = model.encode(references, convert_to_tensor=True)
pred_embeddings = model.encode(predictions, convert_to_tensor=True)

In [ ]:
from sentence_transformers.util import cos_sim

similarities = []

for i in range(len(ref_embeddings)):
    sim = cos_sim(ref_embeddings[i], pred_embeddings[i]).item()
    similarities.append(sim)

print("Average Semantic Similarity:", sum(similarities)/len(similarities))

In [ ]:
print("\n===== Machine Translation Evaluation =====")

print("BLEU:", bleu_score["bleu"])
print("ROUGE-L:", rouge_score["rougeL"])
print("METEOR:", meteor_score["meteor"])
print("chrF:", chrf.score)
print("Embedding Similarity:", sum(similarities)/len(similarities))

In [ ]:
ref_embeddings_np = ref_embeddings.cpu().numpy()
pred_embeddings_np = pred_embeddings.cpu().numpy()

In [ ]:
import numpy as np

np.save("reference_embeddings.npy", ref_embeddings_np)
np.save("prediction_embeddings.npy", pred_embeddings_np)

In [ ]:
import pandas as pd

df_embeddings = pd.DataFrame({
    "reference": references,
    "prediction": predictions
})

df_embeddings["ref_embedding"] = list(ref_embeddings_np)
df_embeddings["pred_embedding"] = list(pred_embeddings_np)

df_embeddings.to_pickle("translation_embeddings.pkl")

In [ ]:
df_embeddings = pd.read_pickle("translation_embeddings.pkl")

TAMIL TRANSLATION EVALUATION

In [1]:
!pip install transformers sentencepiece -q

In [3]:
!pip install indic-transliteration sentencepiece transformers -q

In [4]:
from datasets import load_dataset

dataset = load_dataset("ai4bharat/IndicMTEval", split="test")

print(len(dataset))

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/498k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/200k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.13M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2380 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/4997 [00:00<?, ? examples/s]

2380


In [5]:
dataset_tamil = dataset.filter(lambda x: x["language"] == "Tamil")

print("Tamil samples:", len(dataset_tamil))

Filter:   0%|          | 0/2380 [00:00<?, ? examples/s]

Tamil samples: 276


In [7]:
sources = dataset_tamil["src"]
references = dataset_tamil["ref"]

print(len(sources))
print(sources[0])
print(references[0])

276
The U.N. also hopes to finalize a fund to help countries affected by global warming to cope with the impacts.
உலக வெப்பமயமாதலால் பாதிக்கப்பட்ட நாடுகளுக்கு இந்த பாதிப்புகளை சமாளிக்க உதவும் வகையில் ஒரு நிதியை முடிவு செய்ய இயலும் என U.N. நம்புகிறது.


In [8]:
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

sources = [preprocess(t) for t in sources]
references = [preprocess(t) for t in references]

In [29]:
from transformers import NllbTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = NllbTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [30]:
model_predictions = []

batch_size = 8

for i in range(0, len(sources), batch_size):

    batch = sources[i:i+batch_size]

    inputs = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    translated = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("tam_Taml")
    )

    preds = tokenizer.batch_decode(
        translated,
        skip_special_tokens=True
    )

    model_predictions.extend(preds)

print("Total predictions:", len(model_predictions))

Total predictions: 276


In [31]:
model_predictions = [preprocess(t) for t in model_predictions]

In [32]:
import evaluate
import sacrebleu
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("Evaluating translations...\n")

# BLEU
bleu = evaluate.load("bleu")
references_bleu = [[ref] for ref in references]

bleu_score = bleu.compute(
    predictions=model_predictions,
    references=references_bleu
)

# chrF
chrf = sacrebleu.corpus_chrf(
    model_predictions,
    [references]
)

# BERTScore
bertscore = evaluate.load("bertscore")

bert_results = bertscore.compute(
    predictions=model_predictions,
    references=references,
    lang="ta"
)

bert_f1 = np.mean(bert_results["f1"])

# Embedding Similarity
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

ref_embeddings = embed_model.encode(references)
pred_embeddings = embed_model.encode(model_predictions)

similarities = cosine_similarity(
    ref_embeddings,
    pred_embeddings
).diagonal()

embedding_similarity = np.mean(similarities)

# Print Results
print("===== Evaluation Results =====\n")

print("BLEU Score:", bleu_score["bleu"])
print("chrF Score:", chrf.score)
print("BERTScore F1:", bert_f1)
print("Embedding Similarity:", embedding_similarity)

Evaluating translations...



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


===== Evaluation Results =====

BLEU Score: 0.14073740207540267
chrF Score: 54.626687127913556
BERTScore F1: 0.8497447468664335
Embedding Similarity: 0.8760938


In [33]:
for i in range(5):
    print("SOURCE:", sources[i])
    print("REFERENCE:", references[i])
    print("MODEL:", model_predictions[i])
    print()

SOURCE: the u.n. also hopes to finalize a fund to help countries affected by global warming to cope with the impacts.
REFERENCE: உலக வெப்பமயமாதலால் பாதிக்கப்பட்ட நாடுகளுக்கு இந்த பாதிப்புகளை சமாளிக்க உதவும் வகையில் ஒரு நிதியை முடிவு செய்ய இயலும் என u.n. நம்புகிறது.
MODEL: உலக வெப்பமயமாதலால் பாதிக்கப்பட்ட நாடுகளுக்கு தாக்கங்களை சமாளிக்க உதவும் ஒரு நிதியை இறுதி செய்ய ஐ. நா. நம்புகிறது.

SOURCE: the u.n. also hopes to finalize a fund to help countries affected by global warming to cope with the impacts.
REFERENCE: உலக வெப்பமயமாதலால் பாதிக்கப்பட்ட நாடுகளுக்கு இந்த பாதிப்புகளை சமாளிக்க உதவும் வகையில் ஒரு நிதியை முடிவு செய்ய இயலும் என u.n. நம்புகிறது.
MODEL: உலக வெப்பமயமாதலால் பாதிக்கப்பட்ட நாடுகளுக்கு தாக்கங்களை சமாளிக்க உதவும் ஒரு நிதியை இறுதி செய்ய ஐ. நா. நம்புகிறது.

SOURCE: the u.n. also hopes to finalize a fund to help countries affected by global warming to cope with the impacts.
REFERENCE: உலக வெப்பமயமாதலால் பாதிக்கப்பட்ட நாடுகளுக்கு இந்த பாதிப்புகளை சமாளிக்க உதவும் வகையில் ஒரு நிதிய

google/t5-small

In [11]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-base"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [12]:
t5_predictions = []

batch_size = 8

for i in range(0, len(sources), batch_size):

    batch = sources[i:i+batch_size]

    inputs = tokenizer(
        ["translate English to Tamil: " + s for s in batch],
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    outputs = model.generate(**inputs)

    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    t5_predictions.extend(preds)

print("Total predictions:", len(t5_predictions))

Total predictions: 276


In [14]:
t5_predictions = [preprocess(t) for t in t5_predictions]

In [24]:
!pip install bert-score -q

In [27]:
clean_refs = []
clean_preds = []

for ref, pred in zip(references, t5_predictions):
    if ref.strip() != "" and pred.strip() != "":
        clean_refs.append(ref)
        clean_preds.append(pred)

print("Valid samples:", len(clean_preds))

Valid samples: 270


In [28]:
import evaluate
import sacrebleu
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from bert_score import score

print("Evaluating translations...\n")

# BLEU
bleu = evaluate.load("bleu")

references_bleu = [[ref] for ref in clean_refs]

bleu_score = bleu.compute(
    predictions=clean_preds,
    references=references_bleu
)

# chrF
chrf = sacrebleu.corpus_chrf(
    clean_preds,
    [clean_refs]
)

# BERTScore
P, R, F1 = score(
    clean_preds,
    clean_refs,
    model_type="bert-base-multilingual-cased"
)

bert_f1 = F1.mean().item()

# Embedding similarity
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

ref_embeddings = embed_model.encode(clean_refs)
pred_embeddings = embed_model.encode(clean_preds)

similarities = cosine_similarity(
    ref_embeddings,
    pred_embeddings
).diagonal()

embedding_similarity = np.mean(similarities)

print("\n===== Evaluation Results =====\n")

print("BLEU Score:", bleu_score["bleu"])
print("chrF Score:", chrf.score)
print("BERTScore F1:", bert_f1)
print("Embedding Similarity:", embedding_similarity)

Evaluating translations...



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


===== Evaluation Results =====

BLEU Score: 0.0
chrF Score: 0.22674024250644748
BERTScore F1: 0.6173190474510193
Embedding Similarity: 0.28766814


In [29]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

model_name = "facebook/m2m100_418M"

tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

print("M2M100 model loaded")

tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

M2M100 model loaded


In [31]:
tokenizer.src_lang = "en"

In [32]:
m2m_predictions = []

batch_size = 8

for i in range(0, len(sources), batch_size):

    batch = sources[i:i+batch_size]

    encoded = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.get_lang_id("ta")
    )

    preds = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )

    m2m_predictions.extend(preds)

print("Total predictions:", len(m2m_predictions))

Total predictions: 276


In [33]:
m2m_predictions = [preprocess(t) for t in m2m_predictions]

In [34]:
import evaluate
import sacrebleu
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from bert_score import score

print("Evaluating translations...\n")

# CHANGE THIS depending on model
predictions = m2m_predictions

# Remove empty sentences
clean_refs = []
clean_preds = []

for ref, pred in zip(references, predictions):
    if ref.strip() != "" and pred.strip() != "":
        clean_refs.append(ref)
        clean_preds.append(pred)

print("Valid samples:", len(clean_preds))

# BLEU
bleu = evaluate.load("bleu")

references_bleu = [[ref] for ref in clean_refs]

bleu_score = bleu.compute(
    predictions=clean_preds,
    references=references_bleu
)

# chrF
chrf = sacrebleu.corpus_chrf(
    clean_preds,
    [clean_refs]
)

# BERTScore
P, R, F1 = score(
    clean_preds,
    clean_refs,
    model_type="bert-base-multilingual-cased"
)

bert_f1 = F1.mean().item()

# Embedding Similarity
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

ref_embeddings = embed_model.encode(clean_refs)
pred_embeddings = embed_model.encode(clean_preds)

similarities = cosine_similarity(
    ref_embeddings,
    pred_embeddings
).diagonal()

embedding_similarity = np.mean(similarities)

# Print Results
print("\n===== Evaluation Results =====\n")

print("BLEU Score:", bleu_score["bleu"])
print("chrF Score:", chrf.score)
print("BERTScore F1:", bert_f1)
print("Embedding Similarity:", embedding_similarity)

Evaluating translations...

Valid samples: 276


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



===== Evaluation Results =====

BLEU Score: 0.008726440899402313
chrF Score: 17.461341772346902
BERTScore F1: 0.6680243611335754
Embedding Similarity: 0.5583976
